# Notebook to analyze content of extracellular flexible loop 4 (EL4)

In [1]:
import pandas as pd
import seaborn as sns

EL4.fasta was obtained by opening the multiple sequenced alignment of all Fks homologues (used for the phylogenetic tree) in UGene and saving the subalignment of EL4 as a fasta file

In [2]:
with open('data/EL4.fasta', 'r') as file:
    file_content = file.read()
    
    lines = file_content.split('\n')

headers = [h[1:] for h in lines if h[0]=='>']
sequences = [h for h in lines if h[0]!='>']        

df = pd.DataFrame(list(zip(headers,sequences)), columns=['header','sequence'])
df

,header,sequence
0,Chlamydomonas_reinhardtii_1,GVSGEPGQVS----MTQT
1,Chlamydomonas_reinhardtii_2,GVSGEPGQVS----MTQT
2,Arabidopsis_thaliana_1,L-----ADSTDTNAALGV
3,Arabidopsis_thaliana_3,L-----ADSTDTNAALGV
4,Arabidopsis_thaliana_2,KRSSSNEALG--------
...,...,...
405,Nadsonia_fulvescens,CNYNRNRPISDLLEPIGC
406,Geotrichum_candidum_1,CLYDRNKPITDPHIPLGC
407,Geotrichum_candidum_2,CLYDKNKPLTDPQIPTGC
408,Blastobotrys_adeninivorans,CMYDRNKPITDPQIPVGC


In [3]:
# Check that EL4 is delimited by the two conserved cysteines for all manually selected sequences
df['C_both_sides'] = df.sequence.apply(lambda x: True if (x[0]=='C') & (x[-1]=='C') else False)
df[df.C_both_sides != True]

,header,sequence,C_both_sides
0,Chlamydomonas_reinhardtii_1,GVSGEPGQVS----MTQT,False
1,Chlamydomonas_reinhardtii_2,GVSGEPGQVS----MTQT,False
2,Arabidopsis_thaliana_1,L-----ADSTDTNAALGV,False
3,Arabidopsis_thaliana_3,L-----ADSTDTNAALGV,False
4,Arabidopsis_thaliana_2,KRSSSNEALG--------,False
5,Arabidopsis_thaliana_4,KRSSSNEALG--------,False
58,Puccinia_graminis_1,------------------,False
59,Puccinia_graminis_2,------------------,False
182,Purpureocillium_lilacinum_2,------------------,False


In [4]:
# Length of loop
df['length'] = df.sequence.apply(lambda x: len(x) - x.count('-'))
df

,header,sequence,C_both_sides,length
0,Chlamydomonas_reinhardtii_1,GVSGEPGQVS----MTQT,False,14
1,Chlamydomonas_reinhardtii_2,GVSGEPGQVS----MTQT,False,14
2,Arabidopsis_thaliana_1,L-----ADSTDTNAALGV,False,13
3,Arabidopsis_thaliana_3,L-----ADSTDTNAALGV,False,13
4,Arabidopsis_thaliana_2,KRSSSNEALG--------,False,10
...,...,...,...,...
405,Nadsonia_fulvescens,CNYNRNRPISDLLEPIGC,True,18
406,Geotrichum_candidum_1,CLYDRNKPITDPHIPLGC,True,18
407,Geotrichum_candidum_2,CLYDKNKPLTDPQIPTGC,True,18
408,Blastobotrys_adeninivorans,CMYDRNKPITDPQIPVGC,True,18


In [9]:
# Filter on delimitations and length
df_filtered = df[(df.C_both_sides) & (df.length == 18)].reset_index(drop=True)
df_filtered

,header,sequence,C_both_sides,length,R_count,K_count,RK_count
0,Schizosaccharomyces_japonicus_2,CHYTTSDAINAAFRPSGC,True,18,1,0,1
1,Schizosaccharomyces_japonicus_5,CHYTTSDAINAAFRPSGC,True,18,1,0,1
2,Schizosaccharomyces_octosporus_3,CEYQASGAVNQGLYPSGC,True,18,0,0,0
3,Schizosaccharomyces_octosporus_7,CEYQASGAVNQGLYPSGC,True,18,0,0,0
4,Schizosaccharomyces_pombe_3,CDYQAGAAINASLYPPGC,True,18,0,0,0
...,...,...,...,...,...,...,...
362,Nadsonia_fulvescens,CNYNRNRPISDLLEPIGC,True,18,2,0,2
363,Geotrichum_candidum_1,CLYDRNKPITDPHIPLGC,True,18,1,1,2
364,Geotrichum_candidum_2,CLYDKNKPLTDPQIPTGC,True,18,0,2,2
365,Blastobotrys_adeninivorans,CMYDRNKPITDPQIPVGC,True,18,1,1,2


In [10]:
df_filtered['R_count'] = df_filtered.sequence.str.count('R')
df_filtered['K_count'] = df_filtered.sequence.str.count('K')
df_filtered['RK_count'] = df_filtered.R_count + df_filtered.K_count
df_filtered

,header,sequence,C_both_sides,length,R_count,K_count,RK_count
0,Schizosaccharomyces_japonicus_2,CHYTTSDAINAAFRPSGC,True,18,1,0,1
1,Schizosaccharomyces_japonicus_5,CHYTTSDAINAAFRPSGC,True,18,1,0,1
2,Schizosaccharomyces_octosporus_3,CEYQASGAVNQGLYPSGC,True,18,0,0,0
3,Schizosaccharomyces_octosporus_7,CEYQASGAVNQGLYPSGC,True,18,0,0,0
4,Schizosaccharomyces_pombe_3,CDYQAGAAINASLYPPGC,True,18,0,0,0
...,...,...,...,...,...,...,...
362,Nadsonia_fulvescens,CNYNRNRPISDLLEPIGC,True,18,2,0,2
363,Geotrichum_candidum_1,CLYDRNKPITDPHIPLGC,True,18,1,1,2
364,Geotrichum_candidum_2,CLYDKNKPLTDPQIPTGC,True,18,0,2,2
365,Blastobotrys_adeninivorans,CMYDRNKPITDPQIPVGC,True,18,1,1,2


In [17]:
gby = df_filtered.groupby('sequence')[['header','RK_count']].agg(n_homologues = ('header','nunique'),
                                                           RK_count = ('RK_count', lambda x: x.astype(bool).sum())
                                                          )
gby

,n_homologues,RK_count
sequence,,
CDYDHDQKLTVPMRPEGC,2,2
CDYDHNQKLTVKMRPEGC,2,2
CDYDHNQKLTVPLMPPGC,2,2
CDYDKNKPITDLLKPIGC,1,1
CDYDKNKPITDVLKPYGC,2,2
...,...,...
CYYDKSQKLSTTFVPQGC,2,2
CYYNGNQKLSTSIVPRGC,2,2
CYYNKNQPITDPLVPIGC,1,1


In [19]:
gby[gby.RK_count > 0]

,n_homologues,RK_count
sequence,,
CDYDHDQKLTVPMRPEGC,2,2
CDYDHNQKLTVKMRPEGC,2,2
CDYDHNQKLTVPLMPPGC,2,2
CDYDKNKPITDLLKPIGC,1,1
CDYDKNKPITDVLKPYGC,2,2
...,...,...
CYYDKSQKLSTTFVPQGC,2,2
CYYNGNQKLSTSIVPRGC,2,2
CYYNKNQPITDPLVPIGC,1,1
